In [1]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.messages import ToolMessage
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv
import requests

In [2]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> dict:
    """
    This function fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/1818a743efb787c832926e1c/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value.

    conversion_rate is marked with Annotated + InjectedToolArg
    which means:

    - The LLM will NOT generate this argument.
    - This value is injected programmatically (by us or the system).
    - It is hidden from the model's tool-call schema.
    - Useful when one tool depends on another tool's output.

    In short:
    conversion_rate is system-controlled, not model-controlled.
    """

    return base_currency_value * conversion_rate

In [3]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1771804801,
 'time_last_update_utc': 'Mon, 23 Feb 2026 00:00:01 +0000',
 'time_next_update_unix': 1771891201,
 'time_next_update_utc': 'Tue, 24 Feb 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 90.8652}

In [4]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': 85.16})

851.5999999999999

In [5]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.5
)

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [6]:
messages = [
    SystemMessage(
        content="You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer."
    ),
    HumanMessage('What is the conversion factor between INR and USD, and based on that 10 INR to USD')
]

In [7]:
messages

[SystemMessage(content='You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between INR and USD, and based on that 10 INR to USD', additional_kwargs={}, response_metadata={})]

In [8]:
ai_message = llm_with_tools.invoke(messages)

In [9]:
messages.append(ai_message)

In [10]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'mr9mmn4p3',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': '0.0137'},
  'id': 'n2tdpb3x6',
  'type': 'tool_call'}]

In [11]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call["name"] == "get_conversion_factor":
        tool_message1 = get_conversion_factor.invoke(tool_call["args"])
        
        # fetch this conversion rate
        conversion_rate = tool_message1["conversion_rate"]
        
        # append this tool message to messages list
        messages.append(
            ToolMessage(
                content=str(conversion_rate),
                name="get_conversion_factor",
                tool_call_id=tool_call["id"],
            )
        )

    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call["name"] == "convert":
        # override the conversion_rate with the real one
        tool_call["args"]["conversion_rate"] = conversion_rate
        
        tool_message2 = convert.invoke(tool_call["args"])
        
        messages.append(
            ToolMessage(
                content=str(tool_message2),
                name="convert",
                tool_call_id=tool_call["id"],
            )
        )

In [12]:
messages

[SystemMessage(content='You are a helpful assistant. Use tools only when necessary. After receiving the tool result, respond with the final answer.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between INR and USD, and based on that 10 INR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'mr9mmn4p3', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'n2tdpb3x6', 'function': {'arguments': '{"base_currency_value":10,"conversion_rate":"0.0137"}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 534, 'total_tokens': 583, 'completion_time': 0.083972477, 'completion_tokens_details': None, 'prompt_time': 0.030606628, 'prompt_tokens_details': None, 'queue_time': 0.045455978, 'total_time': 0.114579105}, 'model_name':

In [13]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01101. Therefore, 10 INR is equal to 0.1101 USD.'

##### ChatGPT

In [14]:
# from langchain_core.messages import ToolMessage

# messages.append(ai_message)

# # Step 1: Execute first tool
# tool_call = ai_message.tool_calls[0]

# tool_output = get_conversion_factor.invoke(tool_call["args"])

# rate = tool_output["conversion_rate"]

# messages.append(
#     ToolMessage(
#         content=str(rate),
#         name="get_conversion_factor",
#         tool_call_id=tool_call["id"],
#     )
# )

# # Step 2: Call model again
# second_ai = llm_with_tools.invoke(messages)
# messages.append(second_ai)

# # Step 3: Execute convert tool (inject rate manually)
# if second_ai.tool_calls:
#     second_tool_call = second_ai.tool_calls[0]

#     converted_value = convert.invoke({
#         "base_currency_value": second_tool_call["args"]["base_currency_value"],
#         "conversion_rate": rate
#     })

#     messages.append(
#         ToolMessage(
#             content=str(converted_value),
#             name="convert",
#             tool_call_id=second_tool_call["id"],
#         )
#     )

#     final_ai = llm_with_tools.invoke(messages)
#     print(final_ai.content)

# else:
#     # Model skipped convert tool
#     print("Final Answer:", second_ai.content)

# # Step 4: Final answer
# final_ai = llm_with_tools.invoke(messages)

# print(final_ai.content)